# PRISM ROAD AI — RDD2022 YOLOv8 Pothole Detection Training Pipeline

This Google Colab notebook provides the reproducible training pipeline for fine-tuning YOLOv8 on the **RDD2022 (Road Damage Dataset - India Subset)**.

### Dataset Overview & Damage Classes:
- **D40**: Pothole / Road Cavitation (Primary focus)
- **D00**: Longitudinal Crack
- **D10**: Transverse Crack
- **D20**: Alligator Fatigue Crack

Dataset source: [CRDDC 2022 / Seki Lab Figshare](https://figshare.com/articles/dataset/RDD2022_-_The_multi-national_Road_Damage_Dataset_released_through_CRDDC_2022/21431547)

In [ ]:
# 1. Install Ultralytics and dependencies
!pip install -q ultralytics albumentations pyyaml

In [ ]:
# 2. Check GPU acceleration
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 3. Download RDD2022 India Subset or Load Prepared Samples
!mkdir -p rdd2022_data/images rdd2022_data/labels
# Note: Full India dataset tar archive can be extracted into rdd2022_data/
print("RDD2022 directory initialized ready for XML to YOLO label conversion.")

In [ ]:
# 4. Pascal VOC XML to YOLOv8 Format Converter
import os, xml.etree.ElementTree as ET

CLASS_MAP = {'D40': 0, 'D00': 1, 'D10': 2, 'D20': 3}

def convert_voc_to_yolo(xml_file, out_txt_file, img_w=640, img_h=480):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    lines = []
    for obj in root.findall('object'):
        cname = obj.findtext('name', '').strip()
        if cname in CLASS_MAP:
            cid = CLASS_MAP[cname]
            bnd = obj.find('bndbox')
            xmin = float(bnd.findtext('xmin'))
            ymin = float(bnd.findtext('ymin'))
            xmax = float(bnd.findtext('xmax'))
            ymax = float(bnd.findtext('ymax'))
            xc = ((xmin + xmax) / 2.0) / img_w
            yc = ((ymin + ymax) / 2.0) / img_h
            w = (xmax - xmin) / img_w
            h = (ymax - ymin) / img_h
            lines.append(f"{cid} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
    with open(out_txt_file, 'w') as f:
        f.write('\n'.join(lines))

print("Converter defined successfully.")

In [ ]:
# 5. Create dataset.yaml configuration
yaml_content = '''
path: ./rdd2022_data
train: images/train
val: images/val
names:
  0: pothole
  1: longitudinal_crack
  2: transverse_crack
  3: alligator_crack
'''
with open('rdd2022.yaml', 'w') as f:
    f.write(yaml_content.strip())
print('rdd2022.yaml created.')

In [ ]:
# 6. Train YOLOv8 Model
from ultralytics import YOLO

# Load pretrained YOLOv8 nano / small weights
model = YOLO('yolov8s.pt')

# Train on RDD2022
results = model.train(
    data='rdd2022.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='prism_pothole_yolov8',
    patience=10,
    save=True
)

In [ ]:
# 7. Validate and Report Metrics (Precision, Recall, mAP50, mAP50-95)
metrics = model.val()
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
# 8. Export Weights for PRISM FastAPI Backend
# Copy 'runs/detect/prism_pothole_yolov8/weights/best.pt' into 'prism_backend/models/best.pt'
exported_pt = 'runs/detect/prism_pothole_yolov8/weights/best.pt'
print(f"Trained weights saved at: {exported_pt}")
# Also export ONNX format for lightweight deployment:
model.export(format='onnx', imgsz=640)